# Data Entry

In [ ]:
import pandas as pd  # noqa: F401
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

reload_ = False

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them.
"""


import src.database  # noqa: E402, F401
import src.database.models as models
from src.database import SessionLocal, engine, seed_provincias, seed_admin  # noqa: E402, F401



def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        seed_provincias(db)
        print("Generando usuario maestro...")
        seed_admin(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

In [ ]:
import os
import pandas as pd
from importlib import reload

reset_and_seed()

if not reload_:
    
    print("🗄️ Importando datos de la empresa...")
    print("   ➡️ Importando socios comerciales...")
    import src.imports.init_socios as init_socios
    import src.imports.cql.read as read_files
    print("   ➡️ Importando cliente...")
    import src.imports.cql.clients as clients
    print("   ➡️ Importando creditos...")
    import src.imports.cql.credits as credits
    print("   ➡️ Importando cuotas y cobranzas...")
    import src.imports.cql.quota_and_coll as quotas
    reload_ = True
else:

    print("🗄️ Importando datos de la empresa...")
    print("   ➡️ Importando socios comerciales...")
    reload(init_socios)
    reload(read_files)
    print("   ➡️ Importando cliente...")
    reload(clients)
    print("   ➡️ Importando creditos...")
    reload(credits)
    print("   ➡️ Importando cuotas y cobranzas...")
    reload(quotas)